# NVS Benchmark no Colab - Execucao Unica (Todos os Metodos x Todas as Cenas)

Fluxo one-click no Google Colab.

Como usar:
1. Edite apenas os parametros na Celula 2.
2. Execute somente a Celula 3.
3. Aguarde: o pipeline roda tudo, consolida metricas, gera relatorio geral, cria rankings e exporta artifacts.


In [ ]:
# Celula 2 - Parametros de execucao
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git"
REPO_DIR = "/content/TCC"
BRANCH = "main"

# Cobertura principal: metodos oficiais x datasets principais
OFFICIAL_METHODS = ["nerf_static", "nerf_dynamic", "gs_static", "gs_dynamic"]
PRIMARY_DATASETS = ["blender_synthetic", "d_nerf", "mipnerf360"]

# Para override manual, defina listas; para usar cobertura principal, deixe None
INCLUDE_METHODS = None
INCLUDE_DATASETS = None
EXCLUDED_METHODS = {"external"}

# Configuracao de execucao
PRESET = "quick"
SPLIT = "train"
ADAPTIVE_PRESET = True

# Modo estrito: qualquer falha obrigatoria bloqueia resultado final
STRICT_MODE = True
MIN_REQUIRED_PAIRS = 1
REQUIRE_FINITE_METRICS = True

# Retomada e controle
RESUME_FROM_EXISTING = True
FAIL_FAST = False
MAX_COMBINATIONS = None

# Ambiente
MIN_FREE_DISK_GB = 8.0

# Saida
REPORT_NAME = "colab_full_matrix_report"
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NVS_Benchmark/full_matrix"
AUTO_DOWNLOAD_ZIP = True

In [ ]:
# Celula 3 - Execucao unica end-to-end (strict mode + resultados reais)
import json
import math
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Validacao de ambiente Colab
try:
    __import__("google.colab")
except Exception as exc:
    raise RuntimeError("Este notebook foi desenhado para Google Colab.") from exc


def _is_finite_number(value):
    try:
        return math.isfinite(float(value))
    except Exception:
        return False


def _validate_metric_payload(payload, min_pairs=1, require_finite=True):
    if not isinstance(payload, dict):
        return False, "payload_is_not_dict"

    required_fields = [
        "pairs",
        "psnr",
        "ssim",
        "lpips",
        "fps",
        "vram_gb",
        "train_seconds",
        "inference_seconds",
        "frame_time_ms",
        "latency_p50_ms",
        "latency_p90_ms",
        "latency_p99_ms",
    ]
    missing = [field for field in required_fields if field not in payload]
    if missing:
        return False, f"missing_fields:{','.join(missing)}"

    pairs = payload.get("pairs", 0)
    try:
        pairs_value = float(pairs)
    except Exception:
        return False, "pairs_not_numeric"

    if pairs_value < float(min_pairs):
        return False, f"pairs_below_min:{pairs_value}<{min_pairs}"

    if require_finite:
        invalid = [field for field in required_fields if not _is_finite_number(payload.get(field))]
        if invalid:
            return False, f"non_finite_fields:{','.join(invalid)}"

    return True, ""


def _load_combo_snapshot_metric(snapshot_file, expected_method):
    path = Path(snapshot_file)
    if not path.exists():
        return None, "snapshot_not_found"

    try:
        data = json.loads(path.read_text(encoding="utf-8-sig"))
    except Exception as exc:
        return None, f"snapshot_invalid_json:{exc}"

    if not isinstance(data, dict):
        return None, "snapshot_not_dict"

    if expected_method in data:
        return data.get(expected_method), ""

    # fallback para snapshots com um unico metodo
    if len(data) == 1:
        only_key = next(iter(data.keys()))
        if only_key == expected_method:
            return data.get(only_key), ""

    return None, f"method_not_found_in_snapshot:{expected_method}"


# 1) Clone e setup
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "nvs_benchmark.cli", "status"], check=True)

# 2) Drive opcional
if USE_GOOGLE_DRIVE:
    drive_mod = __import__("google.colab", fromlist=["drive"])
    drive = getattr(drive_mod, "drive")
    drive.mount("/content/drive")
    print("Drive montado.")

# 3) Sanidade de ambiente
free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"Espaco livre em /content: {free_gb:.2f} GB")
if free_gb < float(MIN_FREE_DISK_GB):
    raise RuntimeError(f"Espaco insuficiente em /content: {free_gb:.2f} GB < {MIN_FREE_DISK_GB:.2f} GB")

import torch
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 4) Descoberta de metodos e datasets
from nvs_benchmark.data import SUPPORTED_DATASETS
from nvs_benchmark.methods import build_registry_with_all_methods

registry = build_registry_with_all_methods()
all_methods = [m for m in registry.list_ids() if m not in EXCLUDED_METHODS]

if INCLUDE_METHODS:
    selected_methods = [m for m in all_methods if m in set(INCLUDE_METHODS)]
else:
    selected_methods = [m for m in OFFICIAL_METHODS if m in all_methods]

missing_official = [m for m in OFFICIAL_METHODS if m not in all_methods]
if STRICT_MODE and not INCLUDE_METHODS and missing_official:
    raise RuntimeError(f"Metodos oficiais ausentes no ambiente: {missing_official}")

datasets_available = list(SUPPORTED_DATASETS)
if INCLUDE_DATASETS:
    selected_datasets = [d for d in datasets_available if d in set(INCLUDE_DATASETS)]
else:
    selected_datasets = [d for d in PRIMARY_DATASETS if d in datasets_available]

if not selected_methods:
    raise RuntimeError("Nenhum metodo selecionado apos filtros.")
if not selected_datasets:
    raise RuntimeError("Nenhum dataset selecionado apos filtros.")

print("Metodos selecionados:", selected_methods)
print("Datasets selecionados:", selected_datasets)

# 5) Instalar datasets (best effort)
subprocess.run([
    sys.executable,
    "-m",
    "nvs_benchmark.cli",
    "install",
    "--catalog-file",
    "./configs/install_catalog.json",
    "--only",
    "datasets",
    "--execute",
], check=False)

# 6) Resolver roots de cenas por dataset
# Preferencia por um root que contenha ao menos uma cena valida para o split escolhido.
dataset_base_candidates = {
    "blender_synthetic": [
        "./data/blender_synthetic/nerf_synthetic",
        "./data/blender_synthetic",
        "./data/_smoke/blender",
    ],
    "d_nerf": [
        "./data/d_nerf",
    ],
    "mipnerf360": [
        "./data/mipnerf360",
        "./data/nerf_llff_data",
    ],
    "tanks_and_temples": [
        "./data/tanks_and_temples",
    ],
    "custom": [
        "./data/custom",
    ],
}


def choose_scene_root(base_path: Path, split_name: str):
    if not base_path.exists():
        return None

    transforms_name = f"transforms_{split_name}.json"
    if (base_path / transforms_name).exists():
        return base_path

    direct_scenes = [
        p for p in base_path.iterdir()
        if p.is_dir() and (p / transforms_name).exists()
    ]
    if direct_scenes:
        return sorted(direct_scenes)[0]

    nested = sorted(base_path.rglob(transforms_name))
    if nested:
        return nested[0].parent

    # fallback para datasets estilo images/ quando split nao usa transforms
    if (base_path / "images").exists():
        return base_path

    return None


dataset_roots = {}
missing_dataset_roots = []
for ds in selected_datasets:
    chosen = None
    for candidate in dataset_base_candidates.get(ds, []):
        root = choose_scene_root(Path(candidate), SPLIT)
        if root is not None:
            chosen = str(root)
            break
    dataset_roots[ds] = chosen
    if chosen is None:
        missing_dataset_roots.append(ds)

print("Roots resolvidos:")
for ds, root in dataset_roots.items():
    print(f"- {ds}: {root}")

if STRICT_MODE and missing_dataset_roots:
    raise RuntimeError(f"Dataset root nao encontrado para: {missing_dataset_roots}")

# 7) Rodar matriz completa com retomada
metrics_dir = Path("./artifacts/metrics/full_matrix")
metrics_dir.mkdir(parents=True, exist_ok=True)

combo_logs_dir = Path("./artifacts/logs/full_matrix_combo")
combo_logs_dir.mkdir(parents=True, exist_ok=True)

progress_file = Path("./artifacts/metrics/full_matrix_progress.json")
results = []
executed = 0
reused = 0

if progress_file.exists() and RESUME_FROM_EXISTING:
    try:
        results = json.loads(progress_file.read_text(encoding="utf-8"))
    except Exception:
        results = []

# indice para evitar duplicacao por combinacao
done_index = {}
for item in results:
    key = (item.get("dataset"), item.get("method"))
    if key[0] and key[1]:
        done_index[key] = item

expected_pairs = [(dataset, method) for dataset in selected_datasets for method in selected_methods]
stop_now = False
for dataset, method in expected_pairs:
    if stop_now:
        break

    key = (dataset, method)
    if key in done_index and done_index[key].get("status") in {"ok", "reused"}:
        continue

    root = dataset_roots.get(dataset)
    combo_id = f"{dataset}__{method}"
    snapshot_file = str(metrics_dir / f"{combo_id}.json")

    if not root:
        item = {
            "dataset": dataset,
            "method": method,
            "status": "missing_root",
            "reason": "dataset_root_not_found",
            "snapshot_file": None,
        }
        done_index[key] = item
        if STRICT_MODE:
            stop_now = True
        continue

    if RESUME_FROM_EXISTING and Path(snapshot_file).exists():
        payload, load_error = _load_combo_snapshot_metric(snapshot_file, method)
        if load_error:
            status = "failed"
            reason = load_error
        else:
            ok_payload, validation_error = _validate_metric_payload(
                payload,
                min_pairs=MIN_REQUIRED_PAIRS,
                require_finite=REQUIRE_FINITE_METRICS,
            )
            status = "reused" if ok_payload else "failed"
            reason = "snapshot_already_exists" if ok_payload else validation_error

        item = {
            "dataset": dataset,
            "method": method,
            "status": status,
            "reason": reason,
            "snapshot_file": snapshot_file,
        }
        done_index[key] = item
        if status == "reused":
            reused += 1
        elif STRICT_MODE:
            stop_now = True
        continue

    cmd = [
        sys.executable,
        "-m",
        "nvs_benchmark.cli",
        "method-run",
        "--method",
        method,
        "--dataset",
        dataset,
        "--root",
        root,
        "--split",
        SPLIT,
        "--preset",
        PRESET,
        "--output-dir",
        "./artifacts",
        "--log-dir",
        "./logs",
        "--compute-metrics",
        "--snapshot-file",
        snapshot_file,
    ]

    if ADAPTIVE_PRESET:
        cmd.append("--adaptive-preset")

    if STRICT_MODE:
        cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

    print("Executando:", " ".join(cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)

    combo_stdout = completed.stdout or ""
    combo_stderr = completed.stderr or ""
    combo_log_file = combo_logs_dir / f"{combo_id}.log"
    combo_log_file.write_text(
        "# COMMAND\n"
        + " ".join(cmd)
        + "\n\n# RETURN_CODE\n"
        + str(completed.returncode)
        + "\n\n# STDOUT\n"
        + combo_stdout
        + "\n\n# STDERR\n"
        + combo_stderr,
        encoding="utf-8",
    )

    if completed.returncode == 0 and Path(snapshot_file).exists():
        payload, load_error = _load_combo_snapshot_metric(snapshot_file, method)
        if load_error:
            status = "failed"
            reason = load_error
        else:
            ok_payload, validation_error = _validate_metric_payload(
                payload,
                min_pairs=MIN_REQUIRED_PAIRS,
                require_finite=REQUIRE_FINITE_METRICS,
            )
            status = "ok" if ok_payload else "failed"
            reason = "" if ok_payload else validation_error
    else:
        status = "failed"
        reason = f"method_run_return_code={completed.returncode}; see_log={combo_log_file}"

    item = {
        "dataset": dataset,
        "method": method,
        "status": status,
        "reason": reason,
        "snapshot_file": snapshot_file if Path(snapshot_file).exists() else None,
        "combo_log": str(combo_log_file),
    }
    done_index[key] = item
    executed += 1

    if STRICT_MODE and status != "ok":
        stop_now = True

    if FAIL_FAST and status == "failed":
        stop_now = True

    if MAX_COMBINATIONS is not None and executed >= int(MAX_COMBINATIONS):
        stop_now = True

# normaliza resultados no formato final esperado
def _ordered_results(index_map, pairs):
    ordered = []
    for ds, m in pairs:
        base = {
            "dataset": ds,
            "method": m,
            "status": "skipped",
            "reason": "not_executed",
            "snapshot_file": None,
        }
        base.update(index_map.get((ds, m), {}))
        ordered.append(base)
    return ordered

results = _ordered_results(done_index, expected_pairs)
progress_file.parent.mkdir(parents=True, exist_ok=True)
progress_file.write_text(json.dumps(results, indent=2), encoding="utf-8")

print("Execucoes novas:", executed)
print("Combinacoes reaproveitadas:", reused)
print("Execucoes registradas:", len(results))
print("Progresso salvo:", progress_file)

# 8) Validar cobertura e consolidar snapshot final
combined_snapshot = Path("./artifacts/metrics/colab_full_matrix_combined.json")
combined_snapshot.parent.mkdir(parents=True, exist_ok=True)

merged = {}
failed_or_missing = []
for item in results:
    dataset = item["dataset"]
    method = item["method"]
    status = item.get("status")
    snapshot_file = item.get("snapshot_file")

    if status not in {"ok", "reused"}:
        failed_or_missing.append((dataset, method, status, item.get("reason", "")))
        continue

    payload, load_error = _load_combo_snapshot_metric(snapshot_file, method)
    if load_error:
        failed_or_missing.append((dataset, method, "failed", load_error))
        continue

    ok_payload, validation_error = _validate_metric_payload(
        payload,
        min_pairs=MIN_REQUIRED_PAIRS,
        require_finite=REQUIRE_FINITE_METRICS,
    )
    if not ok_payload:
        failed_or_missing.append((dataset, method, "failed", validation_error))
        continue

    merged_key = f"{dataset}|{method}"
    merged[merged_key] = payload

combined_snapshot.write_text(json.dumps(merged, indent=2), encoding="utf-8")

expected_count = len(expected_pairs)
actual_count = len(merged)
print(f"Cobertura consolidada: {actual_count}/{expected_count}")

if STRICT_MODE:
    if failed_or_missing:
        preview = "\n".join([f"- {d}/{m}: status={s} reason={r}" for d, m, s, r in failed_or_missing[:20]])
        raise RuntimeError(
            "Execucao incompleta em strict mode. Combinacoes invalidas:\n" + preview
        )
    if actual_count != expected_count:
        raise RuntimeError(
            f"Snapshot consolidado incompleto: {actual_count}/{expected_count} combinacoes validas."
        )

if not merged:
    raise RuntimeError("Snapshot consolidado vazio. Nenhuma combinacao gerou metrica valida.")

# 9) Gerar relatorio geral
expected_matrix = [f"{dataset}|{method}" for dataset, method in expected_pairs]
subprocess.run([
    sys.executable,
    "-m",
    "nvs_benchmark.cli",
    "report-generate",
    "--snapshot-file",
    str(combined_snapshot),
    "--output-dir",
    "./artifacts/reports",
    "--report-name",
    REPORT_NAME,
    "--log-dir",
    "./logs",
    "--no-pdf",
    "--strict-snapshot",
    "--expected-matrix",
    ",".join(expected_matrix),
], check=True)

report_html = Path(f"./artifacts/reports/{REPORT_NAME}.html")
print("Relatorio geral:", report_html)

# 10) Score por dataset + score global
import pandas as pd

reports_dir = Path("./artifacts/reports")
reports_dir.mkdir(parents=True, exist_ok=True)

execution_summary_csv = reports_dir / "full_matrix_execution_summary.csv"
pd.DataFrame(results).to_csv(execution_summary_csv, index=False)

metric_rows = []
for key, payload in merged.items():
    dataset, method = key.split("|", 1)
    metric_rows.append(
        {
            "dataset": dataset,
            "method": method,
            "pairs": float(payload.get("pairs", 0.0)),
            "psnr": float(payload.get("psnr", 0.0)),
            "ssim": float(payload.get("ssim", 0.0)),
            "lpips": float(payload.get("lpips", 0.0)),
            "fps": float(payload.get("fps", 0.0)),
            "vram_gb": float(payload.get("vram_gb", 0.0)),
            "train_seconds": float(payload.get("train_seconds", 0.0)),
            "inference_seconds": float(payload.get("inference_seconds", 0.0)),
        }
    )

metrics_df = pd.DataFrame(metric_rows)
if metrics_df.empty:
    raise RuntimeError("Tabela de metricas vazia apos consolidacao.")

metrics_csv = reports_dir / "full_matrix_metrics_table.csv"
metrics_df.to_csv(metrics_csv, index=False)

WEIGHTS = {
    "psnr": 0.35,
    "ssim": 0.25,
    "lpips": 0.25,
    "fps": 0.15,
}


def minmax(series: pd.Series) -> pd.Series:
    s_min = float(series.min())
    s_max = float(series.max())
    if s_max - s_min <= 1e-12:
        return pd.Series([1.0] * len(series), index=series.index)
    return (series - s_min) / (s_max - s_min)


score_frames = []
for dataset_name, g in metrics_df.groupby("dataset", sort=True):
    g = g.copy()
    g["psnr_n"] = minmax(g["psnr"])
    g["ssim_n"] = minmax(g["ssim"])
    g["fps_n"] = minmax(g["fps"])
    g["lpips_n"] = 1.0 - minmax(g["lpips"])
    g["dataset_score"] = (
        WEIGHTS["psnr"] * g["psnr_n"]
        + WEIGHTS["ssim"] * g["ssim_n"]
        + WEIGHTS["lpips"] * g["lpips_n"]
        + WEIGHTS["fps"] * g["fps_n"]
    )
    g["dataset_rank"] = g["dataset_score"].rank(ascending=False, method="dense").astype(int)
    score_frames.append(g)

dataset_scores_df = pd.concat(score_frames, ignore_index=True)
dataset_scores_df = dataset_scores_df.sort_values(["dataset", "dataset_rank", "method"]).reset_index(drop=True)

dataset_scores_csv = reports_dir / "full_matrix_dataset_scores.csv"
dataset_scores_df.to_csv(dataset_scores_csv, index=False)

global_df = (
    dataset_scores_df.groupby("method", as_index=False)["dataset_score"]
    .mean()
    .rename(columns={"dataset_score": "global_score"})
)
global_df["global_rank"] = global_df["global_score"].rank(ascending=False, method="dense").astype(int)
global_df = global_df.sort_values(["global_rank", "method"]).reset_index(drop=True)

global_csv = reports_dir / "full_matrix_global_ranking.csv"
global_df.to_csv(global_csv, index=False)

print("CSV resumo:", execution_summary_csv)
print("CSV metricas:", metrics_csv)
print("CSV score por dataset:", dataset_scores_csv)
print("CSV ranking global:", global_csv)
print("Top ranking global:")
print(global_df.head(10))

# 11) Mostrar relatorio no notebook
display_mod = __import__("IPython.display", fromlist=["IFrame"])
IFrame = getattr(display_mod, "IFrame")
IFrame(src=str(report_html), width=1280, height=760)

# 12) Exportacao final (somente quando strict mode passou)
zip_path = "/content/nvs_benchmark_full_matrix"
archive_file = shutil.make_archive(zip_path, "zip", REPO_DIR, "artifacts")
print("Arquivo zip:", archive_file)

if AUTO_DOWNLOAD_ZIP:
    colab_files_mod = __import__("google.colab", fromlist=["files"])
    files = getattr(colab_files_mod, "files")
    files.download(archive_file)

if USE_GOOGLE_DRIVE:
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    metrics_dst = os.path.join(DRIVE_OUTPUT_DIR, "metrics")
    reports_dst = os.path.join(DRIVE_OUTPUT_DIR, "reports")

    if os.path.exists(metrics_dst):
        shutil.rmtree(metrics_dst)
    if os.path.exists(reports_dst):
        shutil.rmtree(reports_dst)

    shutil.copytree("./artifacts/metrics", metrics_dst)
    shutil.copytree("./artifacts/reports", reports_dst)
    print("Backup no Drive concluido em:", DRIVE_OUTPUT_DIR)

print("Execucao unica concluida com sucesso.")